In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

In [43]:
import numpy as np

In [2]:
#xmlFile='C:/3S/Bochum_WV.xml' # r'C:\3S\Bochum_WV.xml'
xmlFile='C:/3S/MVV_FW.xml' 
tree = ET.parse(xmlFile) # xml.etree.ElementTree.ElementTree

In [3]:
root = tree.getroot() # xml.etree.ElementTree.Element

In [4]:
pm = {c:p for p in root.iter() for c in p}  

In [5]:
tableNames=[]
oldTableName=None
for element in root.iter():
    p = None
    if element in pm:
        p = pm[element]
    if p != root:
        continue
    actTableName=element.tag
    if actTableName != oldTableName:
        tableNames.append(actTableName)
        oldTableName=actTableName
print(tableNames)
    

['AGSN', 'ALLG', 'ALLG_BZ', 'ATMO', 'AVOS', 'AVOS_ROWS', 'BZAG', 'BZAG_BZ', 'CIRC', 'CONT', 'CRGL', 'DATENEBENE', 'DPGR', 'DPGR_BZ', 'DPGR_ROWS', 'DRNP', 'DTRO', 'DTRO_ROWD', 'ELEMENTQUERY', 'ETAM', 'ETAM_ROWS', 'ETAR', 'ETAR_ROWS', 'ETAU', 'ETAU_ROWS', 'FSTF', 'FWBZ', 'FWES', 'FWES_BZ', 'FWVB', 'FWVB_BZ', 'GRAV', 'GTXT', 'HAUS', 'KLAP', 'KLAP_BZ', 'KNOT', 'KNOT_BZ', 'LAYR', 'LFAL', 'LFAL_BZ', 'LFKT', 'LFKT_ROWT', 'LTGR', 'MODELL', 'NRCV', 'NSCH', 'NSCH_BZ', 'PARI', 'PARI_BZ', 'PARV', 'PARZ', 'PARZ_BZ', 'PGPR', 'PGRP', 'PGRP_BZ', 'PGRP_PUMP', 'PGRP_PUMP_BZ', 'PHI1', 'PHI1_ROWT', 'PHI2', 'PHI2_ROWS', 'PHIV', 'PHIV_ROWS', 'POLY', 'PROZESSE', 'PUMD', 'PUMD_ROWT', 'PUMK', 'PUMK_ROWS', 'PUMP', 'PUMP_BZ', 'PVAR', 'PVAR_ROWT', 'PZON', 'QVAR', 'QVAR_ROWT', 'RADD', 'RADD_BZ', 'RART', 'RART_BZ', 'RCON', 'RDIV', 'RDIV_BZ', 'RECT', 'REGP', 'RLVG', 'RLVG_BZ', 'RMES', 'RMES_BZ', 'RMES_DPTS', 'RMUL', 'RMUL_BZ', 'ROHR', 'ROHR_BZ', 'RPLAN', 'RRCT', 'RSLW', 'RSLW_BZ', 'RSTN', 'RSTN_BZ', 'RUES', 'RUES_BZ

In [6]:
dataFrames={}
for tableName in tableNames:
    all_records = []
    for elementRow in root.iter(tag=tableName):
        record = {}
        for elementCol in elementRow:
            record[elementCol.tag] = elementCol.text
        all_records.append(record)
    dataFrames[tableName]=pd.DataFrame(all_records) 

In [7]:
dataFrames['SWVT_ROWT'].ZEIT=dataFrames['SWVT_ROWT'].ZEIT.str.replace(',', '.')
dataFrames['SWVT_ROWT'].W=dataFrames['SWVT_ROWT'].W.str.replace(',', '.')

In [8]:
dataFrames['SWVT_ROWT'].ZEIT=pd.to_numeric(dataFrames['SWVT_ROWT'].ZEIT,errors='coerce') 
dataFrames['SWVT_ROWT'].W=pd.to_numeric(dataFrames['SWVT_ROWT'].W,errors='coerce') 

In [9]:
dfSWVT=pd.merge(dataFrames['SWVT_ROWT'],dataFrames['SWVT'],left_on='fk',right_on='pk')

In [10]:
dfSWVT=dfSWVT.assign(_TimeRank=dfSWVT.groupby(['NAME'])['ZEIT'].rank(method='first'))
dfSWVT.dtypes

W               float64
ZEIT            float64
fk               object
pk_x             object
BESCHREIBUNG     object
INTPOL           object
NAME             object
ZEITOPTION       object
fkDE             object
pk_y             object
_TimeRank       float64
dtype: object

In [11]:
dfSWVT[dfSWVT['NAME'] =='AHON_PUZL']

,W,ZEIT,fk,pk_x,BESCHREIBUNG,INTPOL,NAME,ZEITOPTION,fkDE,pk_y,_TimeRank


=== LFKT ===

In [250]:
dfLFKT=pd.merge(dataFrames['LFKT'],dataFrames['LFKT_ROWT'],left_on='pk',right_on='fk')

In [251]:
dfLFKT.head(5)

,BESCHREIBUNG,INTPOL,NAME,ZEITOPTION,fkDE,pk_x,LF,ZEIT,fk,pk_y
0,Beschreibung Tabelle,0,LfTh,0,5310335261658506550,4843828177877341965,1,NaN,4843828177877341965,4783748497487388700
1,Lastgang f(LFT),0,LFKT,0,5310335261658506550,5416134108033797245,1,0,5416134108033797245,5107407354706890507


In [252]:
dfLFKT=dfLFKT.fillna(0)

In [253]:
dfLFKT.head(5)

,BESCHREIBUNG,INTPOL,NAME,ZEITOPTION,fkDE,pk_x,LF,ZEIT,fk,pk_y
0,Beschreibung Tabelle,0,LfTh,0,5310335261658506550,4843828177877341965,1,0,4843828177877341965,4783748497487388700
1,Lastgang f(LFT),0,LFKT,0,5310335261658506550,5416134108033797245,1,0,5416134108033797245,5107407354706890507


In [254]:
dfLFKT.dtypes

BESCHREIBUNG    object
INTPOL          object
NAME            object
ZEITOPTION      object
fkDE            object
pk_x            object
LF              object
ZEIT            object
fk              object
pk_y            object
dtype: object

In [255]:
dfLFKT['ZEIT']=pd.to_numeric(dfLFKT['ZEIT']) 

In [256]:
dfLFKT['LF']=pd.to_numeric(dfLFKT['LF']) 

In [257]:
dfLFKT.dtypes

BESCHREIBUNG    object
INTPOL          object
NAME            object
ZEITOPTION      object
fkDE            object
pk_x            object
LF               int64
ZEIT             int64
fk              object
pk_y            object
dtype: object

In [258]:
dfLFKT.head(5)

,BESCHREIBUNG,INTPOL,NAME,ZEITOPTION,fkDE,pk_x,LF,ZEIT,fk,pk_y
0,Beschreibung Tabelle,0,LfTh,0,5310335261658506550,4843828177877341965,1,0,4843828177877341965,4783748497487388700
1,Lastgang f(LFT),0,LFKT,0,5310335261658506550,5416134108033797245,1,0,5416134108033797245,5107407354706890507


In [259]:
dfLFKT['ZEIT_RANG']=dfLFKT.groupby(['pk_x'])['ZEIT'].rank(ascending=True)

In [260]:
dfLFKT_gLF=dfLFKT.groupby(['pk_x'], as_index=False).agg({'LF':[np.min,np.max]})

In [261]:
dfLFKT_gLF.head()

pk_x   LF     
                       amin amax
0  4843828177877341965    1    1
1  5416134108033797245    1    1

In [262]:
dfLFKT_gLF.columns= [tup[0]+tup[1] for tup in zip(dfLFKT_gLF.columns.get_level_values(0),dfLFKT_gLF.columns.get_level_values(1))]

In [263]:
dfLFKT_gLF.head()

,pk_x,LFamin,LFamax
0,4843828177877341965,1,1
1,5416134108033797245,1,1


In [264]:
dfLFKT_gLF=dfLFKT_gLF.rename(columns={'LFamin':'LF_min','LFamax':'LF_max'})

In [265]:
dfLFKT_gLF.head()

,pk_x,LF_min,LF_max
0,4843828177877341965,1,1
1,5416134108033797245,1,1


In [266]:
pd.merge(dfLFKT,dfLFKT_gLF,left_on='pk_x',right_on='pk_x')

,BESCHREIBUNG,INTPOL,NAME,ZEITOPTION,fkDE,pk_x,LF,ZEIT,fk,pk_y,ZEIT_RANG,LF_min,LF_max
0,Beschreibung Tabelle,0,LfTh,0,5310335261658506550,4843828177877341965,1,0,4843828177877341965,4783748497487388700,1.0,1,1
1,Lastgang f(LFT),0,LFKT,0,5310335261658506550,5416134108033797245,1,0,5416134108033797245,5107407354706890507,1.0,1,1


=== FWVB ===

In [30]:
dfFWVB=pd.merge(dataFrames['FWVB_BZ'],dataFrames['FWVB'],left_on='fk',right_on='pk')

In [25]:
dfFWVB=dfFWVB[dfFWVB['pk_x'] != '-1']

In [32]:
dfFWVB=pd.merge(dfFWVB,dataFrames['LFKT'],left_on='fkLFKT',right_on='pk')

In [28]:
dfFWVB.count()
dfFWVB.sort_values(by='pk_x')['pk_x'].head(5)

0    4611687738516515315
1    4611752366769535193
2    4611813938169812296
3    4611830155562119166
4    4611830283094028733
Name: pk_x, dtype: object

In [33]:
#dfFWVB.head()
dfFWVB.count()

GEOM_CNLN_I       13930
GEOM_CNLN_K       13930
GRAF              13931
GRAF_CNLN_I       13930
GRAF_CNLN_K       13930
INDLAST           13931
INDLFKT2              1
fk                13931
fkDE_x            13931
fkLFKT            13931
fkLFKT2               1
fkQVAR            13931
fkTEVT                1
pk_x              13931
A                 13931
B                 13931
BESCHREIBUNG_x    13931
C                 13931
DELETED           13931
DPHAUS            13931
DPRLMIN               1
DTMIN             13931
IDREFERENZ        13931
IMBG              13931
INDTR             13931
IPLANUNG          13931
IRFV              13930
KENNUNG           13931
LFK               13931
P1SOLL                1
RHO0              13931
SELECT1           13931
TRS0              13931
TRSK              13931
TVL0              13931
V0                13931
VTYP              13931
W0                13930
fkCONT            13931
fkDE_y            13931
fkFWES                1
fkKI            